# Example 11: Model Selection — ABC-SMC over competing hypotheses

[(GitHub link)](https://github.com/heberlr/UQ_PhysiCell/tree/main/examples/template/ex11_ABC_ModelSelection.ipynb)

ex8 used ABC-SMC to calibrate the parameters of **one** model. Here we use the same
machinery to ask a different question: **which mechanism does the data support?**

The observed data ([ObsData.csv](ObsData.csv), the same spheroid-growth curve
calibrated in [ex9](ex9_Calib_BO.ipynb)) shows the live-cell count saturating over
time and a necrotic population appearing by the end of the experiment. Two classic
explanations compete:

| | hypothesis | rule |
|---|---|---|
| **ModelA** | mechanical feedback | `pressure` *decreases* `cycle entry` |
| **ModelB** | nutrient limitation | `oxygen` *increases* `cycle entry` |
| **ModelC** | both act together | pressure **and** oxygen feedback on `cycle entry` |

ABC-SMC model selection returns a **posterior probability for each model**,
`P(model | data) ∝ P(model) · ∫ P(data | θ, model) P(θ | model) dθ`. That integral
(the marginal likelihood) is a built-in Occam's razor: a richer model that spreads
its prior-predictive mass over a wider range of outcomes is penalised unless the
extra mechanism genuinely improves the fit.

**A note on structure:** everything that defines *what* this example runs — the
three candidate models, their priors, the QoI/distance functions, and the ABC-SMC
options — lives in [uq_script.py](uq_script.py), not in this notebook. That file is
also what actually runs the (multi-hour) calibration on a cluster (`python
uq_script.py <num_workers>`, or via [uq_slurm.sh](uq_slurm.sh)). This notebook
`import`s that setup and focuses on exploring the resulting database — one source
of truth, whether you're launching a real run or just looking at its results.

**What you will learn:**
- How to pass several candidate models to `CalibrationContext` via `abc_options["models"]`
- How each candidate gets its own `model_config` (structure / rules file) and `prior`
- How to read `history.get_model_probabilities()` directly, and the `CandidateModels`
  table UQ-PhysiCell adds to the pyABC database to record each candidate's
  configuration (a plain SQL table, kept separate from pyABC's own schema)
- How ABC-SMC trades goodness-of-fit against model complexity

**Ground truth:** `ObsData.csv` was generated from **ModelA** (see
[GenerateData.ipynb](GenerateData.ipynb)) with `cell_cycle_entry = 1440` min and
`apoptosis_rate = 5.787×10⁻⁵` 1/min. A correct model-selection run should therefore
favour ModelA and *not* be fooled into preferring the more flexible ModelC.

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

from pyabc import visualization
from uq_physicell.database.utils import download_file

# All model/prior/QoI/distance-function/ABC-options setup lives in uq_script.py --
# the same file used for the real cluster run (`python uq_script.py <num_workers>`,
# or uq_slurm.sh on SLURM). This notebook only explores the resulting database, so
# it imports that setup rather than redefining it: one source of truth for "what
# ex11 runs", whichever way you run it.
import uq_script
from uq_script import db_path, dic_real_value

# Download the pre-computed database from zenodo (skip the calibration cell below).
# for file_name in [db_path, "ObsData.csv"]:
#     download_file(file_name)

[2026-09-17 12:02:24,238] INFO:uq_script:🔧 CalibrationContext initialized for ABC-SMC calibration
[2026-09-17 12:02:24,238] INFO:uq_script:📊 Database: ex11_ABC_ModelSelection.db
[2026-09-17 12:02:24,239] INFO:uq_script:🎯 QoIs: ['live_cell_count', 'integrated_dead_cell_count']
[2026-09-17 12:02:24,239] INFO:uq_script:🔍 Model 'ModelA': struc=ModelA, params=['apoptosis_rate', 'cell_cycle_entry'], replicates=5
[2026-09-17 12:02:24,239] INFO:uq_script:🔍 Model 'ModelB': struc=ModelB, params=['apoptosis_rate', 'oxygen_cell_cycle_entry_hfm', 'oxygen_cell_cycle_entry_sat'], replicates=5
[2026-09-17 12:02:24,239] INFO:uq_script:🔍 Model 'ModelC': struc=ModelC, params=['apoptosis_rate', 'cell_cycle_entry', 'oxygen_cell_cycle_entry_hfm', 'oxygen_cell_cycle_entry_sat'], replicates=5
[2026-09-17 12:02:24,239] INFO:uq_script:🔀 ABC-SMC model selection enabled over 3 models
[2026-09-17 12:02:24,239] INFO:uq_script:⚙️ Sampler: multicore with 12 workers


## The three candidate models

Each candidate is a section in [uq_config.ini](uq_config.ini). They differ in the
PhysiCell settings file, the rules file, and which parameters are free. Defined in
[uq_script.py](uq_script.py) (`model_config_A/B/C`, `prior_A/B/C`, `models`):

- **ModelA** — `PhysiCell_settings.xml` + `cell_rules.csv` (`pressure → decreases → cycle entry`).
  Free: `cell_cycle_entry` (cycle-phase duration, min) and `apoptosis_rate`.
- **ModelB** — `PhysiCell_settings_models.xml` + `cell_rules_oxygen.csv`
  (`oxygen → increases → cycle entry`). Base cycle-entry rate is fixed at 0, so
  proliferation is driven entirely by oxygen. Free: `apoptosis_rate` and the
  oxygen response `saturation` / `half_max`.
- **ModelC** — `PhysiCell_settings_models.xml` + `cell_rules_both.csv` (pressure **and**
  oxygen rules). Free: a baseline `cell_cycle_entry` rate **plus** the two oxygen
  response parameters and `apoptosis_rate`.

The priors bracket the ground-truth values where they apply and span a plausible
physiological range for the oxygen-response parameters.

In [2]:
# Peek at the setup imported from uq_script.py (nothing is redefined here).
for spec in uq_script.calib_context.models:
    print(f"{spec.name:>7}: struc={spec.model_config['struc_name']:<12} free params={spec.prior.get_parameter_names()}")

 ModelA: struc=ModelA       free params=['apoptosis_rate', 'cell_cycle_entry']
 ModelB: struc=ModelB       free params=['apoptosis_rate', 'oxygen_cell_cycle_entry_hfm', 'oxygen_cell_cycle_entry_sat']
 ModelC: struc=ModelC       free params=['apoptosis_rate', 'cell_cycle_entry', 'oxygen_cell_cycle_entry_hfm', 'oxygen_cell_cycle_entry_sat']


## Distance functions

Defined in [uq_script.py](uq_script.py) (`_relative_rmse`, `distance_live_cells`,
`distance_dead_cells`, `distance_functions`): one distance per QoI, comparing the
simulated summary (`sim`) to the observed one (`obs`) — pyABC's `(x, x_0)`
convention. `Sum_Dead_Cells` is only measured at the final time point, so the
comparison is restricted to the time points where **both** values are finite,
using a **relative** RMSE that puts the live-cell curve (hundreds of cells) and
the single dead-cell endpoint (~1000 cells) on the same scale without adaptive
weighting.

When the simulation cannot be compared to the data — a crashed run (`sim is None`),
a missing/malformed QoI, or the wrong length because a run stopped early — the
distance returns `np.inf` so pyABC **rejects** that particle. Returning `0.0`
there would be a silent bug: a broken simulation would score a perfect fit and
bias the posterior (and the model probabilities) toward whichever models fail
most often.

## Run ABC-SMC model selection

`uq_script.calib_context` is already built (via `abc_options["models"]`, since
this is a model-selection run — the top-level `model_config` / `prior` arguments
are omitted, each candidate carries its own). `uq_script.abc_options` is tuned for
a real cluster run (`max_populations=5`, `max_simulations=1000`,
`num_replicates` from the `.ini` file); running the cell below directly from a
notebook will therefore take a while. For a smaller/faster check, either lower
those in `uq_script.py` before importing it, or just run the real thing with
`python uq_script.py <num_workers>` (see [uq_slurm.sh](uq_slurm.sh) for a SLURM
job) and skip straight to loading the database below.

In [3]:
# Skip this cell if you already have a populated db_path (downloaded above, or
# produced by a separate run of `python uq_script.py` / uq_slurm.sh).
history = uq_script.run_abc_calibration(calib_context=uq_script.calib_context)
print(f"Completed - {history.n_populations} populations, {history.total_nr_simulations} total simulations")

[2026-09-17 12:02:24,247] INFO:uq_script:🚀 Starting ABC-SMC calibration process
[2026-09-17 12:02:24,247] INFO:uq_script:📊 Database: ex11_ABC_ModelSelection.db
[2026-09-17 12:02:24,248] INFO:uq_script:🎯 Max populations: 5
[2026-09-17 12:02:24,248] INFO:uq_script:🔬 Max simulations: 1000
[2026-09-17 12:02:24,248] INFO:uq_script:⚙️ Setting up sampler...
ABC.Sampler INFO: Parallelize sampling on 2 processes.
[2026-09-17 12:02:24,248] INFO:ABC.Sampler:Parallelize sampling on 2 processes.
[2026-09-17 12:02:24,248] INFO:uq_script:Using nested multicore parallelization: 2 outer processes × 5 inner threads = 10 total
[2026-09-17 12:02:24,248] INFO:uq_script:👥 Setting up population strategy...
[2026-09-17 12:02:24,249] INFO:uq_script:📏 Setting up distance function...
[2026-09-17 12:02:24,249] INFO:uq_script:Using fixed distance weights: [1.0, 1.0]
[2026-09-17 12:02:24,249] INFO:uq_script:🔄 Setting up transition function...
[2026-09-17 12:02:24,249] INFO:uq_script:🎯 Setting up epsilon function...

ArrowInvalid: Could not open Parquet input source '<Buffer>': Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.

## Model probabilities

`history.get_model_probabilities()` gives `P(model | data)` per population. The
final row is the selection result. UQ-PhysiCell also records each candidate's
configuration (including a PhysiCell effective-config fingerprint) in a
`CandidateModels` table -- a plain SQL table, so it stays readable even if your
local pyABC version can't open the rest of the database (see below).
`uq_script.get_model_probabilities()` re-opens `db_path` and returns all three
(`probs`, `models_tbl`, `history`) with pyABC's numeric model index already
renamed to each candidate's `Name`.

In [ ]:
# Re-open the database (works whether you ran the calibration cell above, ran
# uq_script.py/uq_slurm.sh separately, or downloaded the pre-computed db).
#
# If you see "Database has version X, latest format version is Y" here, your
# local pyabc is older than the one that produced db_path -- upgrade it and
# restart the kernel: %pip install --upgrade pyabc
probs, models_tbl, history = uq_script.get_model_probabilities()
display(models_tbl)
display(probs)

final = probs.iloc[-1].sort_values(ascending=False)
winner = final.index[0]
print("\nFinal model probabilities:")
for name, p in final.items():
    print(f"  {name}: {p:.3f}")
print(f"\n=> ABC-SMC favours {winner} (P = {final.iloc[0]:.3f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

probs.plot.bar(stacked=True, ax=axes[0])
axes[0].set_xlabel("population"); axes[0].set_ylabel("P(model | data)")
axes[0].set_title("Model probability across populations")
axes[0].legend(title="model", bbox_to_anchor=(1.0, 1.0))

if history.max_t >= 1:
    visualization.plot_epsilons(history, ax=axes[1], yscale="linear")
    axes[1].set_title("Tolerance (epsilon) across populations")
else:
    axes[1].set_visible(False)
plt.tight_layout()
plt.savefig("ModelSelection_probabilities.svg", format="svg")

## Posterior for the selected model

Conditioned on the winning model, ABC-SMC also gives the parameter posterior. For
ModelA we can check it against the known ground truth; a narrow posterior centred
on the true value means the QoIs identify that parameter well.

In [ ]:
winner_idx = models_tbl.loc[models_tbl["Name"] == winner, "ModelIndex"].iloc[0]
df_post, w = history.get_distribution(m=int(winner_idx), t=history.max_t)
display(df_post.head())

def _wmean_wstd(x, w):
    m = np.average(x, weights=w)
    return m, np.sqrt(np.average((x - m) ** 2, weights=w))

print(f"Parameter recovery for {winner}:")
for param in df_post.columns:
    mean, std = _wmean_wstd(df_post[param].to_numpy(), w)
    true = dic_real_value.get(param)
    if true is not None:
        print(f"  {param}: true={true:.3e}  posterior={mean:.3e} +/- {std:.3e}  (relative error {abs(mean - true) / true * 100:.1f}%)")
    else:
        print(f"  {param}: posterior={mean:.3e} +/- {std:.3e}  (no ground truth)")

refval = {k: v for k, v in dic_real_value.items() if k in df_post.columns}
visualization.plot_kde_matrix(df_post, w, refval=refval or None, refval_color="red")
plt.suptitle(f"Posterior — {winner}", y=1.02)

## Takeaways

- **Model selection is one line of configuration.** `abc_options["models"]` turns a
  single-model `CalibrationContext` into a competition; each candidate keeps its own
  structure, rules file and prior.
- **ABC-SMC balances fit against complexity.** ModelC layers oxygen feedback on top
  of the pressure rule and has two extra free parameters. It is not rewarded for that
  here — the pressure-only data does not need the oxygen mechanism, so the marginal
  likelihood penalises the extra prior volume. This is Bayesian Occam's razor, and
  you get it for free.
- **Identifiability still matters.** If two mechanisms produced indistinguishable
  QoIs the probabilities would stay flat — a legitimate result meaning "this data
  cannot separate these hypotheses". Adding the dead-cell endpoint helps, because
  hypoxia and contact inhibition differ most in how much death they produce.
- **Caveat.** ABC model choice is sensitive to the choice of summary statistics
  (Robert et al., 2011). Comparing *distinct* mechanisms (A vs B) is robust; the
  quantitative penalty on a *nested* extension (A vs C) is more fragile — prefer
  the full trajectory as the QoI and keep priors physiologically tight.
- **One script, two uses.** [uq_script.py](uq_script.py) is the single source of
  truth for the setup — this notebook `import`s it rather than redefining it, so
  a cluster run (`python uq_script.py <num_workers>`) and this notebook's
  exploration can never quietly drift apart.

---